In [267]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import  train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

In [268]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier



In [269]:
from itertools import combinations

In [270]:
df = pd.read_csv('training.csv')
df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value,elo_difference,age_difference,value_difference,is_friendly
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51,-221.0,-0.09,-0.18,0
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69,-34.0,0.00,-0.07,1
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76,417.0,-0.24,0.22,0
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66,-227.0,-4.52,0.09,0
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60,77.0,1.00,-0.09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26,113.0,0.50,-0.33,0
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23,-289.0,-1.10,-0.33,0
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41,-195.0,1.70,-0.16,0
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41,-140.0,-0.20,-0.93,0


In [271]:
to_drop =['home_team_elo', 'away_team_elo', 'home_team_avg_age', 'away_team_avg_age', "home_team_market_value", 'away_team_market_value', 'home_score', 'away_score']

df['avg_elo'] = (df['home_team_elo'] + df['away_team_elo'])/2

In [272]:
conditions = [
    (df['home_score'] > df['away_score']),
    (df['home_score'] == df['away_score']),
    (df['home_score'] < df['away_score'])
]

choices = [2,1,0]
df['match_result'] = np.select(conditions, choices)

In [273]:
df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value,elo_difference,age_difference,value_difference,is_friendly,avg_elo,match_result
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51,-221.0,-0.09,-0.18,0,1589.5,0
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69,-34.0,0.00,-0.07,1,1322.0,0
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76,417.0,-0.24,0.22,0,1414.5,2
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66,-227.0,-4.52,0.09,0,1365.5,0
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60,77.0,1.00,-0.09,0,1570.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26,113.0,0.50,-0.33,0,1262.5,0
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23,-289.0,-1.10,-0.33,0,1581.5,0
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41,-195.0,1.70,-0.16,0,1449.5,0
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41,-140.0,-0.20,-0.93,0,1537.0,0


In [274]:
data_train = df.drop(columns= to_drop).copy()

data_train

,date,home_team,away_team,tournament,neutral,elo_difference,age_difference,value_difference,is_friendly,avg_elo,match_result
0,2004,Bahrain,Saudi Arabia,Gulf Cup,1,-221.0,-0.09,-0.18,0,1589.5,0
1,2004,Bermuda,Barbados,Friendly,0,-34.0,0.00,-0.07,1,1322.0,0
2,2004,Kuwait,Yemen,Gulf Cup,0,417.0,-0.24,0.22,0,1414.5,2
3,2004,Oman,Bahrain,Gulf Cup,1,-227.0,-4.52,0.09,0,1365.5,0
4,2004,United Arab Emirates,Qatar,Gulf Cup,1,77.0,1.00,-0.09,0,1570.5,1
...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,African Cup of Nations,1,113.0,0.50,-0.33,0,1262.5,0
20588,2025,Equatorial Guinea,Algeria,African Cup of Nations,1,-289.0,-1.10,-0.33,0,1581.5,0
20589,2025,Sudan,Burkina Faso,African Cup of Nations,1,-195.0,1.70,-0.16,0,1449.5,0
20590,2025,Gabon,Ivory Coast,African Cup of Nations,1,-140.0,-0.20,-0.93,0,1537.0,0


In [275]:
cycle = 5
friendly = .4
current_year = data_train['date'].max()

In [276]:
previous_years = current_year - data_train['date']
time_weight = .5 ** (previous_years / cycle)

In [277]:
match_type = np.where(df['is_friendly'] == 1, friendly, 1.0)

weights = time_weight * match_type

In [278]:
data_train = data_train.sort_values('date').reset_index(drop= True)

In [279]:
data_train.columns

Index(['date', 'home_team', 'away_team', 'tournament', 'neutral',
       'elo_difference', 'age_difference', 'value_difference', 'is_friendly',
       'avg_elo', 'match_result'],
      dtype='str')

In [280]:
y = data_train.iloc[:, [-1]].copy()
X = df.drop(columns=[
    'home_team_elo', 'away_team_elo', 
    'home_team_avg_age', 'away_team_avg_age',
    'home_team_market_value', 'away_team_market_value',
    'home_score', 'away_score', 
    'match_result',
    'home_team', 'away_team', 'tournament' 
])

In [281]:
tscv = TimeSeriesSplit(n_splits= 5)
fold_accuracies = []

In [282]:
fold_accuracies = []
for train, test in tscv.split(X) :
    X_train, X_test = X.iloc[train], X.iloc[test]
    y_train, y_test = y.iloc[train], y.iloc[test]
    w_train = weights.iloc[train]


    model = XGBClassifier(learning_rate=0.05, max_depth=4,n_estimators=200,min_child_weight=21,subsample=0.8, gamma = .5,objective='multi:softprob')
    model.fit(X_train, y_train, sample_weight = w_train)


    prediction = model.predict(X_test)
    accuracy = accuracy_score(y_test, prediction)
    fold_accuracies.append(accuracy)

fold_accuracies


[0.476981351981352,
 0.4833916083916084,
 0.4801864801864802,
 0.48426573426573427,
 0.46853146853146854]

In [283]:
df2 = pd.read_excel('prediction2.xlsx')
df2

,Team,year,avg_age,z_score,Elo,Group
0,Spain,2026,26.7,4.62,2155,H
1,Argentina,2026,29.1,2.85,2113,J
2,France,2026,27.0,5.87,2062,I
3,England,2026,27.1,5.20,2020,L
4,Brazil,2026,29.2,3.36,1988,C
5,Portugal,2026,28.0,3.71,1984,K
6,Colombia,2026,30.1,0.77,1977,K
7,Netherlands,2026,27.7,2.87,1944,F
8,Ecuador,2026,26.1,1.05,1935,E
9,Germany,2026,28.0,3.60,1925,E


In [284]:
group_stage_matchups = []

for group_name, group_data in df2.groupby('Group'):
    matchups = list(combinations(group_data['Team'], 2))

    group_stage_matchups.extend(matchups)

matches_df = pd.DataFrame(group_stage_matchups, columns=['home_team', 'away_team'])
matches_df

,home_team,away_team
0,Mexico,South Korea
1,Mexico,Czechia
2,Mexico,South Africa
3,South Korea,Czechia
4,South Korea,South Africa
...,...,...
67,England,Panama
68,England,Ghana
69,Croatia,Panama
70,Croatia,Ghana


In [285]:
matches_df = matches_df.merge(df2, left_on= 'home_team', right_on = 'Team')
matches_df = matches_df.rename(columns={'Elo': 'home_elo', 'avg_age': 'home_age', 'z_score': 'home_value'})
matches_df = matches_df.drop(columns=['Team', 'year'])


matches_df = matches_df.merge(df2, left_on= 'away_team', right_on = 'Team')
matches_df = matches_df.rename(columns={'Elo': 'away_elo', 'avg_age': 'away_age', 'z_score': 'away_value'})
matches_df = matches_df.drop(columns=['Team', 'year'])

In [286]:
matches_df['Group'] = matches_df['Group_x']
matches_df.drop(columns= ['Group_x', 'Group_y'], inplace= True)

matches_df

,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group
0,Mexico,South Korea,27.9,0.31,1875,28.1,0.09,1758,A
1,Mexico,Czechia,27.9,0.31,1875,27.6,0.30,1740,A
2,Mexico,South Africa,27.9,0.31,1875,26.8,-0.28,1518,A
3,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1740,A
4,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A
...,...,...,...,...,...,...,...,...,...
67,England,Panama,27.1,5.20,2020,30.4,-0.34,1734,L
68,England,Ghana,27.1,5.20,2020,27.0,0.49,1510,L
69,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L
70,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L


In [287]:
matches_df.to_csv('matches.csv', index= False)

In [288]:
matches_df = pd.read_excel('matches_with_cities.xlsx')

In [289]:
matches_df

,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group,City,neutral
0,Mexico,South Korea,27.9,0.31,1875,28.1,0.09,1758,A,Guadalajara,0
1,Mexico,Czechia,27.9,0.31,1875,27.6,0.30,1740,A,Mexico City,0
2,Mexico,South Africa,27.9,0.31,1875,26.8,-0.28,1518,A,Mexico City,0
3,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1740,A,Guadalajara,1
4,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A,Monterrey,1
...,...,...,...,...,...,...,...,...,...,...,...
67,England,Panama,27.1,5.20,2020,30.4,-0.34,1734,L,New York/New Jersey,1
68,England,Ghana,27.1,5.20,2020,27.0,0.49,1510,L,Boston,1
69,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L,Toronto,1
70,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L,Philadelphia,1


In [290]:
cities = [matches_df['City'].unique()]
print(list(cities))

[<ArrowStringArray>
[           'Guadalajara',            'Mexico City',              'Monterrey',
                'Atlanta',              'Vancouver',            'Los Angeles',
 'San Francisco Bay Area',                'Toronto',                'Seattle',
    'New York/New Jersey',                  'Miami',           'Philadelphia',
                 'Boston',            'Kansas City',                'Houston',
                 'Dallas']
Length: 16, dtype: str]


In [291]:
team1 = matches_df['home_team'].unique()
team2 = matches_df['away_team'].unique()

teams = set(team1).union(set(team2))

teams

{'Algeria',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bosnia and Herzegovina',
 'Brazil',
 'Canada',
 'Cape Verde',
 'Colombia',
 'Congo',
 'Croatia',
 'Cura‚àö√üao',
 'Czechia',
 'Ecuador',
 'Egypt',
 'England',
 'France',
 'Germany',
 'Ghana',
 'Haiti',
 'Iran',
 'Iraq',
 'Ivory Coast',
 'Japan',
 'Jordan',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Norway',
 'Panama',
 'Paraguay',
 'Portugal',
 'Qatar',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'South Africa',
 'South Korea',
 'Spain',
 'Sweden',
 'Switzerland',
 'Tunisia',
 'Turkey',
 'United States',
 'Uruguay',
 'Uzbekistan'}

In [292]:
city_temp = {
    'Guadalajara': 'Hot',
    'Mexico City': 'Hot',
    'Monterrey': 'Hot',
    'Miami': 'Hot',
    'Houston': 'Hot',
    'Dallas': 'Hot',
    'Atlanta': 'Hot',
    'Kansas City': 'Neutral',
    'Los Angeles': 'Neutral',
    'San Francisco Bay Area': 'Neutral',
    'New York/New Jersey': 'Mild',
    'Philadelphia': 'Mild',
    'Boston': 'Mild',
    'Seattle': 'Mild',
    'Vancouver': 'Mild',
    'Toronto': 'Mild'
}



team_temp = {
    'Algeria': 'Hot',
    'Argentina': 'Hot',
    'Australia': 'Hot',
    'Austria': 'Mild',
    'Belgium': 'Mild',
    'Bosnia and Herzegovina': 'Neutral',
    'Brazil': 'Hot',
    'Canada': 'Mild',
    'Cape Verde': 'Hot',
    'Colombia': 'Hot',
    'Congo': 'Hot',
    'Croatia': 'Neutral',
    'Curaçao': 'Hot',
    'Czechia': 'Mild',
    'Ecuador': 'Hot',
    'Egypt': 'Hot',
    'England': 'Mild',
    'France': 'Neutral',
    'Germany': 'Mild',
    'Ghana': 'Hot',
    'Haiti': 'Hot',
    'Iran': 'Neutral',
    'Iraq': 'Hot',
    'Ivory Coast': 'Hot',
    'Japan': 'Neutral',
    'Jordan': 'Hot',
    'Mexico': 'Hot',
    'Morocco': 'Hot',
    'Netherlands': 'Mild',
    'New Zealand': 'Mild',
    'Norway': 'Mild',
    'Panama': 'Hot',
    'Paraguay': 'Hot',
    'Portugal': 'Neutral',
    'Qatar': 'Hot',
    'Saudi Arabia': 'Hot',
    'Scotland': 'Mild',
    'Senegal': 'Hot',
    'South Africa': 'Neutral',
    'South Korea': 'Neutral',
    'Spain': 'Neutral',
    'Sweden': 'Mild',
    'Switzerland': 'Mild',
    'Tunisia': 'Hot',
    'Turkey': 'Neutral',
    'United States': 'Neutral',
    'Uruguay': 'Hot',
    'Uzbekistan': 'Neutral'
}

In [293]:
def get_climate_elo_adjustment(team, city):

    c_temp = city_temp.get(city)
    t_temp = team_temp.get(team)
    
    if c_temp == 'Hot':
        if t_temp == 'Hot': return 25     
        elif t_temp == 'Mild': return -15 
    elif c_temp == 'Mild':
        if t_temp == 'Mild': return 15  
        
    return 0

In [294]:
matches_df['home_team'] = matches_df['home_team'].str.strip()
matches_df['away_team'] = matches_df['away_team'].str.strip()
matches_df['City'] = matches_df['City'].str.strip()

In [295]:
matches_df['home_weather_boost'] = matches_df.apply(lambda row: get_climate_elo_adjustment(row['home_team'], row['City']), axis=1)
matches_df['away_weather_boost'] = matches_df.apply(lambda row: get_climate_elo_adjustment(row['away_team'], row['City']), axis=1)


In [296]:
matches_df

,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group,City,neutral,home_weather_boost,away_weather_boost
0,Mexico,South Korea,27.9,0.31,1875,28.1,0.09,1758,A,Guadalajara,0,25,0
1,Mexico,Czechia,27.9,0.31,1875,27.6,0.30,1740,A,Mexico City,0,25,-15
2,Mexico,South Africa,27.9,0.31,1875,26.8,-0.28,1518,A,Mexico City,0,25,0
3,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1740,A,Guadalajara,1,0,-15
4,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A,Monterrey,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,England,Panama,27.1,5.20,2020,30.4,-0.34,1734,L,New York/New Jersey,1,15,0
68,England,Ghana,27.1,5.20,2020,27.0,0.49,1510,L,Boston,1,15,0
69,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L,Toronto,1,0,0
70,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L,Philadelphia,1,0,0


In [297]:
matches_df['home_elo'] = matches_df['home_elo'] + matches_df['home_weather_boost']
matches_df['away_elo'] = matches_df['away_elo'] + matches_df['away_weather_boost']

matches_df


,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group,City,neutral,home_weather_boost,away_weather_boost
0,Mexico,South Korea,27.9,0.31,1900,28.1,0.09,1758,A,Guadalajara,0,25,0
1,Mexico,Czechia,27.9,0.31,1900,27.6,0.30,1725,A,Mexico City,0,25,-15
2,Mexico,South Africa,27.9,0.31,1900,26.8,-0.28,1518,A,Mexico City,0,25,0
3,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1725,A,Guadalajara,1,0,-15
4,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A,Monterrey,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,England,Panama,27.1,5.20,2035,30.4,-0.34,1734,L,New York/New Jersey,1,15,0
68,England,Ghana,27.1,5.20,2035,27.0,0.49,1510,L,Boston,1,15,0
69,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L,Toronto,1,0,0
70,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L,Philadelphia,1,0,0


In [298]:
matches_df.columns

Index(['home_team', 'away_team', 'home_age', 'home_value', 'home_elo',
       'away_age', 'away_value', 'away_elo', 'Group', 'City', 'neutral',
       'home_weather_boost', 'away_weather_boost'],
      dtype='str')

In [299]:
data_train.columns

Index(['date', 'home_team', 'away_team', 'tournament', 'neutral',
       'elo_difference', 'age_difference', 'value_difference', 'is_friendly',
       'avg_elo', 'match_result'],
      dtype='str')

In [300]:
matches_df['elo_difference'] = matches_df['home_elo'] - matches_df['away_elo']
matches_df['age_difference'] = matches_df['home_age'] - matches_df['away_age']
matches_df['value_difference'] = matches_df['home_value'] - matches_df['away_value']
matches_df['is_friendly'] = 0
matches_df['avg_elo'] = (matches_df['home_elo'] + matches_df['away_elo'])/2
date = 2026
matches_df.insert(0, 'date', date)
matches_df

,date,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group,City,neutral,home_weather_boost,away_weather_boost,elo_difference,age_difference,value_difference,is_friendly,avg_elo
0,2026,Mexico,South Korea,27.9,0.31,1900,28.1,0.09,1758,A,Guadalajara,0,25,0,142,-0.2,0.22,0,1829.0
1,2026,Mexico,Czechia,27.9,0.31,1900,27.6,0.30,1725,A,Mexico City,0,25,-15,175,0.3,0.01,0,1812.5
2,2026,Mexico,South Africa,27.9,0.31,1900,26.8,-0.28,1518,A,Mexico City,0,25,0,382,1.1,0.59,0,1709.0
3,2026,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1725,A,Guadalajara,1,0,-15,33,0.5,-0.21,0,1741.5
4,2026,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A,Monterrey,1,0,0,240,1.3,0.37,0,1638.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026,England,Panama,27.1,5.20,2035,30.4,-0.34,1734,L,New York/New Jersey,1,15,0,301,-3.3,5.54,0,1884.5
68,2026,England,Ghana,27.1,5.20,2035,27.0,0.49,1510,L,Boston,1,15,0,525,0.1,4.71,0,1772.5
69,2026,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L,Toronto,1,0,0,174,-2.1,1.47,0,1821.0
70,2026,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L,Philadelphia,1,0,0,398,1.3,0.64,0,1709.0


In [301]:
matches_df.to_csv('mtm.csv')

In [302]:
matches_df.columns

Index(['date', 'home_team', 'away_team', 'home_age', 'home_value', 'home_elo',
       'away_age', 'away_value', 'away_elo', 'Group', 'City', 'neutral',
       'home_weather_boost', 'away_weather_boost', 'elo_difference',
       'age_difference', 'value_difference', 'is_friendly', 'avg_elo'],
      dtype='str')

In [303]:
X_pred = matches_df[['date', 
                'neutral', 
                'elo_difference', 
                'age_difference', 
                'value_difference', 
                'is_friendly',
                'avg_elo']]


probabilities = model.predict_proba(X=X_pred)

matches_df['prob_away_win'] = probabilities[:, 0]
matches_df['prob_draw'] = probabilities[:, 1]
matches_df['prob_home_win'] = probabilities[:, 2]


matches_df

,date,home_team,away_team,home_age,home_value,home_elo,away_age,away_value,away_elo,Group,...,home_weather_boost,away_weather_boost,elo_difference,age_difference,value_difference,is_friendly,avg_elo,prob_away_win,prob_draw,prob_home_win
0,2026,Mexico,South Korea,27.9,0.31,1900,28.1,0.09,1758,A,...,25,0,142,-0.2,0.22,0,1829.0,0.336265,0.157607,0.506128
1,2026,Mexico,Czechia,27.9,0.31,1900,27.6,0.30,1725,A,...,25,-15,175,0.3,0.01,0,1812.5,0.345182,0.166131,0.488687
2,2026,Mexico,South Africa,27.9,0.31,1900,26.8,-0.28,1518,A,...,25,0,382,1.1,0.59,0,1709.0,0.276751,0.187337,0.535912
3,2026,South Korea,Czechia,28.1,0.09,1758,27.6,0.30,1725,A,...,0,-15,33,0.5,-0.21,0,1741.5,0.285111,0.195583,0.519305
4,2026,South Korea,South Africa,28.1,0.09,1758,26.8,-0.28,1518,A,...,0,0,240,1.3,0.37,0,1638.0,0.300343,0.166045,0.533611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026,England,Panama,27.1,5.20,2035,30.4,-0.34,1734,L,...,15,0,301,-3.3,5.54,0,1884.5,0.348557,0.185547,0.465896
68,2026,England,Ghana,27.1,5.20,2035,27.0,0.49,1510,L,...,15,0,525,0.1,4.71,0,1772.5,0.306544,0.223901,0.469555
69,2026,Croatia,Panama,28.3,1.13,1908,30.4,-0.34,1734,L,...,0,0,174,-2.1,1.47,0,1821.0,0.310696,0.212580,0.476724
70,2026,Croatia,Ghana,28.3,1.13,1908,27.0,0.49,1510,L,...,0,0,398,1.3,0.64,0,1709.0,0.290480,0.201018,0.508501


In [304]:
X

,date,neutral,elo_difference,age_difference,value_difference,is_friendly,avg_elo
0,2004,1,-221.0,-0.09,-0.18,0,1589.5
1,2004,0,-34.0,0.00,-0.07,1,1322.0
2,2004,0,417.0,-0.24,0.22,0,1414.5
3,2004,1,-227.0,-4.52,0.09,0,1365.5
4,2004,1,77.0,1.00,-0.09,0,1570.5
...,...,...,...,...,...,...,...
20587,2025,1,113.0,0.50,-0.33,0,1262.5
20588,2025,1,-289.0,-1.10,-0.33,0,1581.5
20589,2025,1,-195.0,1.70,-0.16,0,1449.5
20590,2025,1,-140.0,-0.20,-0.93,0,1537.0


In [325]:
result_df = matches_df[['home_team', 'away_team', 'Group', 'prob_home_win', "prob_draw", 'prob_away_win']]


result_df

,home_team,away_team,Group,prob_home_win,prob_draw,prob_away_win
0,Mexico,South Korea,A,0.506128,0.157607,0.336265
1,Mexico,Czechia,A,0.488687,0.166131,0.345182
2,Mexico,South Africa,A,0.535912,0.187337,0.276751
3,South Korea,Czechia,A,0.519305,0.195583,0.285111
4,South Korea,South Africa,A,0.533611,0.166045,0.300343
...,...,...,...,...,...,...
67,England,Panama,L,0.465896,0.185547,0.348557
68,England,Ghana,L,0.469555,0.223901,0.306544
69,Croatia,Panama,L,0.476724,0.212580,0.310696
70,Croatia,Ghana,L,0.508501,0.201018,0.290480


In [326]:
result_df['prob_away_win'] = (result_df['prob_away_win'] * 100).round(2).astype(str)
result_df['prob_home_win'] = (result_df['prob_home_win'] * 100).round(2).astype(str)
result_df['prob_draw'] = (result_df['prob_draw'] * 100).round(2).astype(str)

In [327]:
result_df

,home_team,away_team,Group,prob_home_win,prob_draw,prob_away_win
0,Mexico,South Korea,A,50.61,15.76,33.63
1,Mexico,Czechia,A,48.87,16.61,34.52
2,Mexico,South Africa,A,53.59,18.73,27.68
3,South Korea,Czechia,A,51.93,19.56,28.51
4,South Korea,South Africa,A,53.36,16.6,30.03
...,...,...,...,...,...,...
67,England,Panama,L,46.59,18.55,34.86
68,England,Ghana,L,46.96,22.39,30.65
69,Croatia,Panama,L,47.67,21.26,31.07
70,Croatia,Ghana,L,50.85,20.1,29.05


In [328]:
change = { 'Cura‚àö√üao' : "Curaçao"}

result_df["home_team"] = result_df["home_team"].replace(change)
result_df["away_team"] = result_df["away_team"].replace(change)

In [329]:
result_df['prob_away_win'] = result_df['away_team'] + ' : ' + result_df['prob_away_win'] + ' %'
result_df['prob_home_win'] = result_df['home_team'] + ' : ' + result_df['prob_home_win'] + ' %'
result_df['prob_draw'] = result_df['prob_draw'] + ' %'

In [330]:
result_df

,home_team,away_team,Group,prob_home_win,prob_draw,prob_away_win
0,Mexico,South Korea,A,Mexico : 50.61 %,15.76 %,South Korea : 33.63 %
1,Mexico,Czechia,A,Mexico : 48.87 %,16.61 %,Czechia : 34.52 %
2,Mexico,South Africa,A,Mexico : 53.59 %,18.73 %,South Africa : 27.68 %
3,South Korea,Czechia,A,South Korea : 51.93 %,19.56 %,Czechia : 28.51 %
4,South Korea,South Africa,A,South Korea : 53.36 %,16.6 %,South Africa : 30.03 %
...,...,...,...,...,...,...
67,England,Panama,L,England : 46.59 %,18.55 %,Panama : 34.86 %
68,England,Ghana,L,England : 46.96 %,22.39 %,Ghana : 30.65 %
69,Croatia,Panama,L,Croatia : 47.67 %,21.26 %,Panama : 31.07 %
70,Croatia,Ghana,L,Croatia : 50.85 %,20.1 %,Ghana : 29.05 %


In [331]:
result_df.columns

Index(['home_team', 'away_team', 'Group', 'prob_home_win', 'prob_draw',
       'prob_away_win'],
      dtype='str')

In [332]:
result_df.rename(columns= {'home_team' : 'Team1', 'away_team' : 'Team2', 'prob_home_win' : 'Team 1 Win', 'prob_draw' : 'Draw', 'prob_away_win' : 'Team 2 win'}, inplace= True)

result_df.to_csv('almost.csv', index= False)

In [333]:
# result_df = matches_df[['home_team', 'away_team', 'Group']]

# result_df.to_csv('almost.csv', index= False)

In [334]:
dd = pd.read_csv('prediction.csv')

dd

,Team,year,avg_age,z_score,Elo
0,Spain,2026,26.7,4.62,2155
1,Argentina,2026,29.1,2.85,2113
2,France,2026,27.0,5.87,2062
3,England,2026,27.1,5.20,2020
4,Brazil,2026,29.2,3.36,1988
5,Portugal,2026,28.0,3.71,1984
6,Colombia,2026,30.1,0.77,1977
7,Netherlands,2026,27.7,2.87,1944
8,Ecuador,2026,26.1,1.05,1935
9,Germany,2026,28.0,3.60,1925
